# Objective:

This project aims to use deep learning technologies to recognize the emotions from voice interactions. As a possible extra activity in the project, We would like to explore the possibility of having a model trained on a single language dataset being tested for multiple languages, understanding how predicted emotions compare in feature information and overall similarity.


## Project Background: 

Speech Emotion Recognition can be an incredible tool, used during interaction between human to human (H2H), human to machine(H2M), and Machine to Machine (M2M),  attempting to recognize emotion and affective states from speech. 

Human voice, through tone, pitch and velocity, often reflects underlying emotions and can convey a lot of context during interactions. Even animals are equipped with capabilities to recognize emotions from voice and general human expressions. 

As part of speech recognition, this aspect is incredibly important for systems that can deal with humans in a variety of situations: 
telemarketing and customer service
shopping assistants
emergency response 
patient care and health advice

Analyzing voice features to identify changes in emotions can be an important tool to prevent interactions from escalating, allowing a better understanding of situational context that can impact action planning in professional services settings.


In [6]:
# install dependencies
!pip install -q --disable-pip-version-check awswrangler pyathena
!pip install -q --upgrade boto3 botocore awscli

In [7]:
import boto3
import sagemaker
import pandas as pd
import awswrangler as wr
from pyathena import connect
import math
import os
import librosa
import librosa.display
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from datasets import load_dataset

In [8]:
# download CREMA-D dataset from S3 bucket
!aws s3 cp s3://ser-multilingual-spr2026msaai540-group2/CREMAD.zip .

download: s3://ser-multilingual-spr2026msaai540-group2/CREMAD.zip to ./CREMAD.zip


In [17]:
# create data directory
!mkdir -p ~/datasets/CREMAD

In [23]:
# find the exact path of the download zip on SageMaker JupyterLab space
!find / -name "CREMAD.zip" 2>/dev/null

In [18]:
# use the correct zip path to download the unzip the audio files
ZIP_PATH="/mnt/custom-file-systems/efs/fs-0edfdaae5597678e1_fsap-01998c72e6468329e/CREMAD.zip"
!unzip -q "$ZIP_PATH" -d ~/datasets/CREMAD

In [21]:
# check data usage
!du -sh ~/datasets/CREMAD
!df -h

592M	/home/sagemaker-user/datasets/CREMAD
Filesystem      Size  Used Avail Use% Mounted on
overlay          37G   72M   37G   1% /
tmpfs            64M     0   64M   0% /dev
shm             1.8G   28K  1.8G   1% /dev/shm
/dev/nvme0n1p1  180G   94G   87G  52% /opt/.sagemakerinternal
/dev/nvme1n1     25G  847M   25G   4% /home/sagemaker-user
127.0.0.1:/     8.0E   32M  8.0E   1% /mnt/custom-file-systems/efs/fs-0edfdaae5597678e1_fsap-01998c72e6468329e
tmpfs           7.8G     0  7.8G   0% /proc/acpi
tmpfs           7.8G     0  7.8G   0% /sys/firmware


In [22]:
# find the AudioWav folder
!find ~/datasets/CREMAD -maxdepth 4 -type d -name "AudioWAV"

/home/sagemaker-user/datasets/CREMAD/AudioWAV


In [24]:
# define the audiofile directory
AUDIO_DIR = "/home/sagemaker-user/datasets/CREMAD/AudioWAV"